In [1]:
%env AWS_PROFILE=platform-developer

env: AWS_PROFILE=platform-developer


In [45]:
import sys
import os
import logging
from lxml import etree
from adapters.extractors.oai_pmh.axiell.runtime import AXIELL_CONFIG
from adapters.extractors.oai_pmh.axiell import config

client = AXIELL_CONFIG.build_oai_client()

axiell_raw_records = {}

for i, record in enumerate(client.list_records(metadata_prefix="oai_raw", set_spec="collect>museum")):
    try:
        axiell_raw_records[record.header.identifier] = record
    except Exception as e:
        print(e)

    if i % 10000 == 0:
        print(i)

BadArgumentError: Set 'museum' does not exist.

In [19]:
from pathlib import Path
from typing import Any
import lxml.etree as ET
from utils.marc import parse_single_marc_record

STYLESHEET_PATH = "/Users/brychtas/Documents/GitHub/axiell-collections-xslt/data/stylesheets/axc_to_marcxml_collect.xsl"

PARSER = ET.XMLParser(remove_blank_text=True)
XSLT_DOC = ET.parse(STYLESHEET_PATH, PARSER)
TRANSFORM = ET.XSLT(XSLT_DOC)

def prepare_for_xslt(record: Any) -> ET._Element:
    for elem in record.iter():
        elem.tag = ET.QName(elem).localname
    adlib = ET.Element("adlibXML")
    record_list = ET.SubElement(adlib, "recordList")
    record_list.append(record)
    return adlib


def apply_xslt(
    xml_doc: Any,
    **xslt_params: Any,
):
    formatted_params = {
        key: ET.XSLT.strparam(str(value)) for key, value in xslt_params.items()
    }
    return TRANSFORM(xml_doc, **formatted_params)


In [39]:
import pickle
import polars as pl

AXIELL_MARC_WORKS_PATH = "/Users/brychtas/Documents/work-snapshots/axiell-raw-works.parquet"

def save_parquet_snapshot(data: dict, path: str):
    filtered = {k: v for k, v in data.items() if v.metadata is not None}
    print(len(data), len(filtered))
    df = pl.DataFrame({
        "id": list(filtered.keys()),
        "body": [pickle.dumps(etree.tostring(v.metadata)) for v in filtered.values()],
    })
    df.write_parquet(path)

In [20]:
from io import BytesIO
from pymarc import XMLWriter, parse_xml_to_array

axiell_marc_works = {}

xslt_result = apply_xslt(prepare_for_xslt(record.metadata), control_003="UkLW")
record = parse_xml_to_array(BytesIO(bytes(xslt_result)))[0]

In [40]:
save_parquet_snapshot(axiell_raw_records, AXIELL_MARC_WORKS_PATH)

228495 227899


In [42]:

record = client.get_record(identifier=f"collect:110009887", metadata_prefix="oai_raw")
print(etree.tostring(record.metadata, pretty_print=True, encoding="unicode"))


<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" priref="110009887" created="2026-05-16T03:40:44Z" modification="2026-05-16T03:40:44Z" selected="false" deleted="false">
  <accession_status>
    <value lang="neutral">OPEN</value>
  </accession_status>
  <current_location.barcode/>
  <current_location.context>215/215;B11/215;B11;MR/215;B11;MR;108/215;B11;MR;108;6/215;B11;MR;108;6;4</current_location.context>
  <current_location.lref>4385</current_location.lref>
  <current_location.name>215;B11;MR;108;6;4</current_location.name>
  <current_location.package_location>
    <value lang="neutral">LOCATION</value>
    <value lang="0">location</value>
    <value lang="1">standplaats</value>
    <value lang="3">Standort</value>
    <value lang="9">plats</value>
    <value lang="10">מיקום</value>
    <value lang="11">placering</value>
    <value lang="14">位置</value>
  </current_location.package_location>
  <dataset_name>
    <value lang="ne